In [9]:
import ee
import geemap
# NOTA: "viirs-peru" es el ID del proyecto de Google Cloud vinculado a Earth Engine, el nombre fue ese pero es para todo EARTH ENGINE
# sirve como autenticación para CUALQUIER dataset de Earth Engine (WorldCover, SRTM,
# Sentinel-2, Open Buildings, etc.), no solo para VIIRS.

try:
    ee.Initialize(project="viirs-peru")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="viirs-peru")

print("Google Earth Engine listo")

Google Earth Engine listo


In [10]:
from pathlib import Path
import geopandas as gpd

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

geobase = gpd.read_file(DATA / "clean" / "staging" / "geobase_distrital.gpkg")
print("Distritos en geobase:", len(geobase))

Distritos en geobase: 1889


## Fuente: JRC Global Surface Water (JRC/GSW1_4/YearlyHistory)

**Qué es:** Clasificación anual de superficie de agua a 30m de resolución,
construida desde el archivo histórico de Landsat (1984-presente). Clasifica
cada píxel en: sin dato, no agua, agua estacional, agua permanente.

**Qué mide:** Presencia física de cuerpos de agua superficial. No mide agua
potable ni acceso a red de agua entubada.

**Por qué importa para anemia:** Proxy de dos factores del cuello de botella
de agua: (a) disponibilidad de agua para consumo informal o riego en zonas
sin red, y (b) exposición a fuentes de agua no tratada. Complementa —no
reemplaza— la variable de cuerpos de agua que saldrá de la segmentación en
Etapa 2: JRC da la serie histórica confiable a nivel distrital ahora;
la segmentación dará la ubicación exacta relativa a cada caserío.

**Variables extraídas:**
- `pct_agua_permanente` — fracción del área distrital con agua permanente
- `pct_agua_estacional` — fracción del área distrital con agua estacional

**Método de agregación:** Remapeo de la clase categórica a dos máscaras
binarias (permanente / estacional), luego `ee.Reducer.mean()` sobre cada
máscara → fracción de píxeles de esa clase sobre el total del polígono.

**Temporalidad:** Año único reciente (aprox. estable año a año en zona
rural, salvo eventos extremos). Une al maestro solo por `ubigeo`, igual
que SRTM.

**Fuente de polígonos:** `limite_distrital` (shapefile INEI 2025),
`reduceRegions` en lotes de 40 distritos.

In [11]:
jrc_coleccion = ee.ImageCollection("JRC/GSW1_4/YearlyHistory")

# Ver todos los años disponibles
anios_disponibles = jrc_coleccion.aggregate_array("system:index").getInfo()
print(anios_disponibles)

['1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021']


In [12]:
# ============================================================
# JRC Global Surface Water — extracción de agua permanente/estacional por distrito
# ============================================================

# ------------------------------------------------------------
# 1. Verificar último año disponible en el dataset
# ------------------------------------------------------------

jrc_coleccion = ee.ImageCollection("JRC/GSW1_4/YearlyHistory")

anios_disponibles = jrc_coleccion.aggregate_array("system:index").getInfo()
ultimo_anio_str = anios_disponibles[-1]
ANIO_JRC = int(ultimo_anio_str)
print(f"Usando año: {ANIO_JRC}")

# ------------------------------------------------------------
# 2. Cargar la imagen del año elegido y construir máscaras binarias
# ------------------------------------------------------------

jrc = jrc_coleccion \
    .filter(ee.Filter.calendarRange(ANIO_JRC, ANIO_JRC, "year")) \
    .first()

# La banda se llama "waterclass":
# 0 = sin dato, 1 = no agua, 2 = agua estacional, 3 = agua permanente

banda = jrc.select("waterClass")

mask_permanente = banda.eq(3).rename("permanente")
mask_estacional = banda.eq(2).rename("estacional")

# Combinar en una sola imagen de 2 bandas para reducir juntas
img_jrc = mask_permanente.addBands(mask_estacional)

# ------------------------------------------------------------
# 3. Función de reducción por lote
# ------------------------------------------------------------

def procesar_lote_jrc(fc_lote):
    """Aplica reduceRegions sobre un lote de distritos para JRC."""
    resultado = img_jrc.reduceRegions(
        collection=fc_lote,
        reducer=ee.Reducer.mean(),
        scale=30,  # resolución nativa de JRC
        tileScale=4
    )
    return resultado

# ------------------------------------------------------------
# 4. Procesamiento por lotes de 40 distritos
# ------------------------------------------------------------

# geobase es GeoDataFrame — la columna viene como "UBIGEO" (mayúsculas)
lista_ubigeos = geobase["UBIGEO"].tolist()
n_distritos = len(lista_ubigeos)
tamano_lote = 40

resultados_jrc = []

for i in range(0, n_distritos, tamano_lote):
    lote_ids = lista_ubigeos[i:i + tamano_lote]

    # Filtrar el lote en geopandas
    gdf_lote = geobase[geobase["UBIGEO"].isin(lote_ids)]

    # Convertir SOLO este lote a ee.FeatureCollection
    fc_lote = geemap.geopandas_to_ee(gdf_lote)

    fc_resultado = procesar_lote_jrc(fc_lote)

    datos_lote = fc_resultado.reduceColumns(
        ee.Reducer.toList(3),
        ["UBIGEO", "permanente", "estacional"]
    ).get("list").getInfo()

    resultados_jrc.extend(datos_lote)
    print(f"Lote {i // tamano_lote + 1}/{-(-n_distritos // tamano_lote)} — "
          f"distritos {i} a {min(i + tamano_lote, n_distritos)} listo")

# ------------------------------------------------------------
# 5. Armar el dataframe final
# ------------------------------------------------------------

df_jrc = pd.DataFrame(
    resultados_jrc,
    columns=["ubigeo", "pct_agua_permanente", "pct_agua_estacional"]
)

print(df_jrc.shape)
print(df_jrc.describe())
df_jrc.head()

Usando año: 2021
Lote 1/48 — distritos 0 a 40 listo
Lote 2/48 — distritos 40 a 80 listo
Lote 3/48 — distritos 80 a 120 listo
Lote 4/48 — distritos 120 a 160 listo
Lote 5/48 — distritos 160 a 200 listo
Lote 6/48 — distritos 200 a 240 listo
Lote 7/48 — distritos 240 a 280 listo
Lote 8/48 — distritos 280 a 320 listo
Lote 9/48 — distritos 320 a 360 listo
Lote 10/48 — distritos 360 a 400 listo
Lote 11/48 — distritos 400 a 440 listo
Lote 12/48 — distritos 440 a 480 listo
Lote 13/48 — distritos 480 a 520 listo
Lote 14/48 — distritos 520 a 560 listo
Lote 15/48 — distritos 560 a 600 listo
Lote 16/48 — distritos 600 a 640 listo
Lote 17/48 — distritos 640 a 680 listo
Lote 18/48 — distritos 680 a 720 listo
Lote 19/48 — distritos 720 a 760 listo
Lote 20/48 — distritos 760 a 800 listo
Lote 21/48 — distritos 800 a 840 listo
Lote 22/48 — distritos 840 a 880 listo
Lote 23/48 — distritos 880 a 920 listo
Lote 24/48 — distritos 920 a 960 listo
Lote 25/48 — distritos 960 a 1000 listo
Lote 26/48 — distritos

NameError: name 'pd' is not defined

In [13]:
import pandas as pd

In [14]:
df_jrc = pd.DataFrame(
    resultados_jrc,
    columns=["ubigeo", "pct_agua_permanente", "pct_agua_estacional"]
)

print(df_jrc.shape)
print(df_jrc.describe())
df_jrc.head()

(1806, 3)
       pct_agua_permanente  pct_agua_estacional
count          1806.000000          1806.000000
mean              0.213432             0.569356
std               0.218605             0.253046
min               0.000000             0.000000
25%               0.020846             0.372273
50%               0.144211             0.574918
75%               0.345324             0.763620
max               1.000000             1.000000


,ubigeo,pct_agua_permanente,pct_agua_estacional
0,010101,0.019574,0.448590
1,010102,0.000000,0.000000
2,010103,0.146739,0.581834
3,010104,0.000000,1.000000
4,010105,0.000000,0.000000


# Descargar

In [15]:
output_path = DATA / "clean" / "staging" / "jrc_distrital.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_jrc.to_csv(output_path, index=False)
print("Guardado en:", output_path)

Guardado en: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\jrc_distrital.csv
